# 02 - API Server
Loads the saved model and runs a FastAPI server. Run `01_Train_Model` first and keep this notebook open while testing.

## 1. Install libraries (run once)

In [ ]:
%pip install -q fastapi "uvicorn[standard]" scikit-learn pandas joblib pydantic

## 2. Imports

In [ ]:
import json
import threading
import time
from pathlib import Path
from typing import Literal

import joblib
import pandas as pd
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, ConfigDict, Field

## 3. Load the saved model

In [ ]:
MODEL_PATH = Path("models/loan_model.joblib")
META_PATH = Path("models/metadata.json")

# Stop early with a clear message if the model hasn't been trained yet
if not MODEL_PATH.exists():
    raise FileNotFoundError("Model not found. Run 01_Train_Model.ipynb first (same folder).")

model = joblib.load(MODEL_PATH)
meta = json.loads(META_PATH.read_text())
print("Loaded model:", meta["best_model"])

## 4. Input and output formats

In [ ]:
# What the API accepts for one person
class LoanApplication(BaseModel):
    model_config = ConfigDict(json_schema_extra={
        "example": {"Has_Patches": 1, "Has_MS": 0, "Owner_Annual_Income_Lakhs": 4.5}
    })
    Has_Patches: Literal[0, 1] = Field(..., description="1 if property has patches, else 0")
    Has_MS: Literal[0, 1] = Field(..., description="1 if MS annotation present, else 0")
    Owner_Annual_Income_Lakhs: float = Field(..., ge=0, le=10_000, description="Annual income in lakhs")


# What the API returns for one person
class Prediction(BaseModel):
    requires_loan: bool
    probability: float
    risk_band: Literal["LOW", "MEDIUM", "HIGH"]
    model: str


# Many people at once
class BatchRequest(BaseModel):
    applications: list[LoanApplication] = Field(..., min_length=1, max_length=1000)


class BatchPrediction(BaseModel):
    count: int
    predictions: list[Prediction]

## 5. Create the API

In [ ]:
app = FastAPI(title="Loan Prediction API",
              description="Predicts whether a customer requires a loan.",
              version="2.0.0")

# Allow calls from any website or app
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


# Turn a probability into LOW / MEDIUM / HIGH
def risk_band(p: float) -> str:
    return "LOW" if p < 0.33 else "MEDIUM" if p < 0.66 else "HIGH"


# Run the model on a list of applications
def run_model(apps: list[LoanApplication]) -> list[Prediction]:
    data = pd.DataFrame([a.model_dump() for a in apps])[meta["features"]]
    probs = model.predict_proba(data)[:, 1]
    return [Prediction(requires_loan=bool(p >= 0.5),
                       probability=round(float(p), 4),
                       risk_band=risk_band(float(p)),
                       model=meta["best_model"])
            for p in probs]


# Check the API is alive
@app.get("/health")
def health():
    return {"status": "ok"}


# Show which model is used
@app.get("/model-info")
def model_info():
    return meta


# Predict for one person
@app.post("/predict", response_model=Prediction)
def predict(application: LoanApplication):
    return run_model([application])[0]


# Predict for many people
@app.post("/predict/batch", response_model=BatchPrediction)
def predict_batch(req: BatchRequest):
    preds = run_model(req.applications)
    return BatchPrediction(count=len(preds), predictions=preds)

## 6. Start the server
Then open http://127.0.0.1:8000/docs in your browser to try it.

In [ ]:
# Run the server in a background thread so the notebook stays usable
config = uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

# Wait until the server is ready
while not server.started:
    time.sleep(0.1)

print("Server running at http://127.0.0.1:8000/docs")

## 7. Stop the server

In [ ]:
# Shut the server down
server.should_exit = True
thread.join(timeout=5)
print("Server stopped")